In [1]:
import torch
import os

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# 1. GPU / CPU auswählen
# ---------------------------------------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple GPU wird benutzt.")
else:
    device = torch.device("cpu")
    print("CPU wird benutzt.")

print("Device:", device)


# ---------------------------------------------------------
# 2. Daten importieren
# ---------------------------------------------------------

def create_sample():
    sample = []

    jubaea_directory = "Data/Jubaea"

    for image in os.listdir(jubaea_directory):
        if image.lower().endswith((".jpg", ".jpeg", ".png")):
            sample.append(
                (os.path.join(jubaea_directory, image), 1)
            )

    not_jubaea_directory = "Data/Not_Jubaea"

    for subfolder in os.listdir(not_jubaea_directory):

        subfolder_path = os.path.join(
            not_jubaea_directory,
            subfolder
        )

        if not os.path.isdir(subfolder_path):
            continue

        for image in os.listdir(subfolder_path):

            if image.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                sample.append(
                    (
                        os.path.join(
                            not_jubaea_directory,
                            subfolder,
                            image
                        ),
                        0
                    )
                )

    return sample


sample = create_sample()

pictures = [x[0] for x in sample]
labels = [x[1] for x in sample]


X_train, X_test, Y_train, Y_test = train_test_split(
    pictures,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)


# ---------------------------------------------------------
# 3. Transformationen
# ---------------------------------------------------------

# Werte, die beim ursprünglichen Training
# von ResNet18 auf ImageNet benutzt wurden
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Training:
# Resize + Data Augmentation + Normalisierung
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomRotation(15),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


# Test:
# KEINE zufällige Data Augmentation
# aber dieselbe Normalisierung
test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


# ---------------------------------------------------------
# 4. Dataset
# ---------------------------------------------------------

class JubaeaDataset(Dataset):

    def __init__(
        self,
        paths,
        labels,
        transform=None
    ):

        self.paths = paths
        self.labels = labels
        self.transform = transform


    def __len__(self):
        return len(self.paths)


    def __getitem__(self, index):

        path = self.paths[index]

        label = self.labels[index]

        image = Image.open(path).convert("RGB")

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = JubaeaDataset(
    X_train,
    Y_train,
    transform=train_transform
)

test_dataset = JubaeaDataset(
    X_test,
    Y_test,
    transform=test_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# ---------------------------------------------------------
# 5. ResNet18 laden
# ---------------------------------------------------------

net = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)


# Bereits trainiertes Jubaea-Modell laden
net.load_state_dict(
    torch.load(
        "Jubaea_resnet.pth",
        map_location=device
    )
)


# GANZ WICHTIG:
# Modell auf GPU verschieben
net = net.to(device)


# ---------------------------------------------------------
# 6. Aktuelles gespeichertes Modell testen
# ---------------------------------------------------------

net.eval()

pre_correct = 0
pre_total = 0


with torch.no_grad():

    for images, labels in test_loader:

        # Daten auf GPU verschieben
        images = images.to(device)
        labels = labels.to(device)

        outputs = net(images)

        preds = torch.argmax(
            outputs,
            dim=1
        )

        pre_correct += (
            preds == labels
        ).sum().item()

        pre_total += labels.size(0)


model_acc = pre_correct / pre_total

print(
    "Accuracy vom geladenen Modell:",
    model_acc
)


# ---------------------------------------------------------
# 7. Lossfunktion
# ---------------------------------------------------------

loss_fn = nn.CrossEntropyLoss()


# ---------------------------------------------------------
# 8. Optimizer
# ---------------------------------------------------------

optimizer = torch.optim.Adam(
    net.parameters(),
    lr=0.0001,
    weight_decay=5e-4
)


# ---------------------------------------------------------
# 9. Training
# ---------------------------------------------------------

epochs = 20

best_acc = model_acc

classes = [
    "Not_Jubaea",
    "Jubaea"
]


for epoch in range(epochs):

    # ---------------------------
    # TRAINING
    # ---------------------------

    net.train()

    loss_calc = 0.0

    correct_train = 0
    total_train = 0


    for images, labels in train_loader:

        # Bilder und Labels auf die GPU
        images = images.to(device)
        labels = labels.to(device)

        # Alte Gradienten löschen
        optimizer.zero_grad()

        # Forward Pass
        outputs = net(images)

        # Loss berechnen
        loss = loss_fn(
            outputs,
            labels
        )

        # Backpropagation
        loss.backward()

        # Gewichte verändern
        optimizer.step()

        loss_calc += loss.item()


        # Trainingsaccuracy berechnen

        preds = torch.argmax(
            outputs,
            dim=1
        )

        correct_train += (
            preds == labels
        ).sum().item()

        total_train += labels.size(0)


    epoch_loss = (
        loss_calc /
        len(train_loader)
    )

    train_acc = (
        correct_train /
        total_train
    )


    print()
    print(
        f"Epoch {epoch + 1}/{epochs}"
    )

    print(
        "Trainingsloss:",
        epoch_loss
    )

    print(
        "Trainingsaccuracy:",
        train_acc
    )


    # ---------------------------
    # TEST
    # ---------------------------

    net.eval()

    correct_test = 0
    total_test = 0


    with torch.no_grad():

        for images, labels in test_loader:

            # Auch Testdaten müssen auf die GPU
            images = images.to(device)
            labels = labels.to(device)

            outputs = net(images)

            probs = torch.softmax(
                outputs,
                dim=1
            )

            preds = torch.argmax(
                outputs,
                dim=1
            )

            correct_test += (
                preds == labels
            ).sum().item()

            total_test += labels.size(0)


            # ---------------------------------------
            # Fehlklassifikationen anzeigen
            # Bei vielen Epochs besser deaktivieren
            # ---------------------------------------

            """
            for i in range(len(images)):

                if preds[i] != labels[i]:

                    # Bild zurück auf CPU,
                    # weil matplotlib keine MPS-Tensoren versteht
                    image = images[i].cpu()

                    # Normalisierung rückgängig machen,
                    # damit das Bild normal aussieht
                    mean = torch.tensor(
                        imagenet_mean
                    ).view(3, 1, 1)

                    std = torch.tensor(
                        imagenet_std
                    ).view(3, 1, 1)

                    image = image * std + mean

                    # Werte sicher auf 0-1 begrenzen
                    image = torch.clamp(
                        image,
                        0,
                        1
                    )

                    plt.imshow(
                        image.permute(1, 2, 0)
                    )

                    plt.title(
                        f"Falsch\n"
                        f"Echte Klasse: "
                        f"{classes[labels[i].item()]}\n"
                        f"Vorhersage: "
                        f"{classes[preds[i].item()]}\n"
                        f"P(Jubaea)="
                        f"{probs[i][1].item():.3f}"
                    )

                    plt.axis("off")
                    plt.show()
            """


    test_acc = (
        correct_test /
        total_test
    )


    # ---------------------------
    # Bestes Modell speichern
    # ---------------------------

    if test_acc > best_acc:

        best_acc = test_acc

        torch.save(
            net.state_dict(),
            "Jubaea_resnet.pth"
        )

        print(
            "Neues bestes Modell gespeichert."
        )


    print(
        "Testaccuracy:",
        test_acc
    )

    print(
        "Beste Accuracy bisher:",
        best_acc
    )

Apple GPU wird benutzt.
Device: mps
Accuracy vom geladenen Modell: 0.411214953271028

Epoch 1/20
Trainingsloss: 0.16751956194639206
Trainingsaccuracy: 0.9578454332552693
Neues bestes Modell gespeichert.
Testaccuracy: 0.9345794392523364
Beste Accuracy bisher: 0.9345794392523364

Epoch 2/20
Trainingsloss: 0.05166336348546403
Trainingsaccuracy: 0.9836065573770492
Neues bestes Modell gespeichert.
Testaccuracy: 0.9532710280373832
Beste Accuracy bisher: 0.9532710280373832

Epoch 3/20
Trainingsloss: 0.02820944892508643
Trainingsaccuracy: 0.9953161592505855
Neues bestes Modell gespeichert.
Testaccuracy: 0.9719626168224299
Beste Accuracy bisher: 0.9719626168224299

Epoch 4/20
Trainingsloss: 0.032221230146075995
Trainingsaccuracy: 0.9953161592505855
Testaccuracy: 0.9626168224299065
Beste Accuracy bisher: 0.9719626168224299

Epoch 5/20
Trainingsloss: 0.01813575000103031
Trainingsaccuracy: 0.9929742388758782
Testaccuracy: 0.9532710280373832
Beste Accuracy bisher: 0.9719626168224299

Epoch 6/20
Tra